In [13]:
import math
import torch

class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias = bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias=None

    def forward(self,x):
        self.x = x
        out = x @ self.weights.T 
        if self.has_bias:
            out = out+self.bias

        return out

    def backward(self,grad_out):
        x_shape = self.x.shape
        flat_x = self.x.flatten(0,-2)
        flat_grad = grad_out.flatten(0,-2)

        grad_inputs = flat_grad @ self.weights
        self.weights.grad = flat_grad.T @ flat_x
        if self.has_bias:
            self.bias.grad = flat_grad.sum(dim=0)
    
        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs
        

class Softmax:
    def forward(self, scores):
        max_score = torch.max(scores, dim=-1, keepdim=True).values
        scores_exp = torch.exp(scores - max_score)
        self.scores_sum = scores_exp.sum(dim=-1, keepdim=True)
        self.out = scores_exp / self.scores_sum
        return self.out
    
    def backward(self, grad_attn):
        grad_scores = self.out * (grad_attn - (grad_attn * self.out).sum(dim=-1, keepdim=True))
        return grad_scores


class CausalSelfAttention:
    def __init__(self, num_heads, num_dims):
        self.num_heads = num_heads
        self.num_dims = num_dims
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims, num_dims, bias=False)
        self.w_k = LinearLayer(num_dims, num_dims, bias=False)
        self.w_v = LinearLayer(num_dims, num_dims, bias=False)
        self.proj_out = LinearLayer(num_dims, num_dims)
        self.softmax = Softmax() 

    def forward(self, x):
        B, T, D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B, T, self.num_heads, self.head_dims).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dims).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dims).transpose(1, 2)

        self.Q = Q
        self.K = K
        self.V = V 
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(masks, float("-inf"))

        self.attn_scores = self.softmax.forward(scores)

        out = self.attn_scores @ self.V 

        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.proj_out.forward(out)
        return out

    def backward(self, grad_out):
        grad = self.proj_out.backward(grad_out)
        B, T, D = grad.shape
        grad = grad.view(B, T, self.num_heads, self.head_dims).transpose(1, 2)

        grad_attn = grad @ self.V.transpose(-2, -1)  
        grad_V = self.attn_scores.transpose(-2, -1) @ grad

        grad_scores = self.softmax.backward(grad_attn) 

        grad_Q = grad_scores @ self.K
        grad_K = grad_scores.transpose(-2, -1) @ self.Q

        grad_Q = grad_Q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_K.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_V.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]


if __name__ == "__main__":
    num_dims = 512
    num_heads = 8
    attention = CasualSelfAttention(num_heads, num_dims)

    batch_size = 4
    seq_len = 10
    x = torch.randn(batch_size, seq_len, num_dims)

    output = attention.forward(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected shape: [{batch_size}, {seq_len}, {num_dims}]")

Input shape: torch.Size([4, 10, 512])
Output shape: torch.Size([4, 10, 512])
Expected shape: [4, 10, 512]



attention ->it try to say , on which parts does the input should focus on

"The cat sat on the mat"
for example we can say in this way
when processing the word cat,it have the most attention to the cat,because cat (who sat)
medium attention -> mat (where)
less attention -> the

this is the core part of the whole attention

firstly
there is a book on the table
so this is an example 
so the Q and K relationship is a matrix
where the shape is (T,T)
we can see how does word gives attention to the other word in that matrix
so we can visualize them in the

```
          there    is      a      book 
there     ....     ....   ...     .....
is        ....     ....   ...     .....
a         ....     ....   ...     .....
book      ....     ....   ...     .....

``` 
these dots represent the scores of how much one word gives attention to another
as i don't know either 😂️ so it's better to take dots .....

we can see this attention and then
the query may ask in this way,take this as an example only,i don't know how exactly it might ask 😂️
Q -> i need a book (usually they aren't in this plain english,but to understand for a bit,we can assume just to make sure,to know,what exactly it was fetching for ?)
as all the keys are gonna say something right,so it would be more attetntion score to the last key book compared to those others and then that last one have the high attention than these others
so in this way they will reresent the Q and K relationship
and then are we done? not yet
because to this part we just did discovered which have the more attention to the each query
and then,what we need ?? we need the value right

so once the moment the scores are found,we reach out for the values section where this
whole matrix is multiplied by the values ,so this is what we do 
(Q @ K.T/sqrt(d_k)) @ V right
so dont need to think of the terminology for now
we can come back later for them
so this is what the attention is 

 
to this point we know the Q K V 
and we need to  think into the inside of this matrix
so how do we find the scores between these
we can see these values right before us,and when we go inside them
we will be introduces to the three 
weights -> w_q w_k w_v
so what exactly are these,and we are just one level above the basement ,which we have the tokens only
so this is the next layer to that basement,as we know about the token right?
and then,how does they contribute to the scores? i mean,yeah we know token did exist but at what kind of attention do they provide each other is the layer 3 and then
we are that at second layer which actually brings us to that layer 3
and ,this is where the weights are introduces as they are already trained on the datatset and then they can find the meaningful projections ,at first they are all random and the moment they are trained and they are made into the better ways of the forming the queries and then keys and then values
we have already taken one example right ,there is a book under the table
and,what if the sentence is changed and then how can we project them now
so these weights are trained in a way that,they are the secret extra ingredients to the food we are baking
so,when the inputs are multiplied as these weights are trained ,so now these inputs
can be formed to ask the queries and then keys and ,even values so that
now we have the clear map of the how do the values are represented
because we need to generate the text too right,not in this step but will soon introduce it too
and then ,with those we are converting the base tokens to the meaningful representations 
and then now we get the Q K and then we will use the softmax and then make them sum upto to the 1 and then we will multiply with the V to get the values 

Why do we need multiple heads?

Imagine the sentence has several relationships at once. One attention mechanism might learn to focus on one kind of relationship
while another can learn a different one. Multi-head attention gives the model several attention mechanisms operating in parallel.


in the multi head attention,we can take the multiple heads and then each head works on its own and then at the end we mix all of what they did find out

this is the order i did followed ,and the reaons are below
so, the first parameter is the num_dims because we need to say in how many dimensions we are gonna projecting these weights
and then next we need to give the number of head so that they will split into match the number of dimensions right

as the head_dims is like
in the entire dimensions ,how many dimensions does a single head gets
its like dividing the items into the equal number of pieces for all heads
as multiple heads can project over different and then we can get more information

now the Q K V can be attained by multiplying the weights with the x 
as we turn off the bias,we don't actually need them here,because it was like,just adds some threshold to the scores and that doesn't make anything diff,so its redundant here so we just tru to find the linear layer calculations without the bias
and once we get the Q K V
the next step is,now the actual shape of the Q -> (B,T,D)
as the B specifies the Batch,gpu can handle multiple batches so we use the batches of the tokens
as that B represents the batches
and the T represents the tokens which we have taken an example right
and the D is the number of dimensions
as we have splitted them into the heads and head_dims
we need to view that Q as the B T num_heads head_dim and then we need to transpose this Q matrix because ,we don't perform any kind of the operation on the head we perfom on the tokens and then dimensions of the heads
we make transpose the dimensions of the num_heads to the Tokens
and now it would looks like
B num_heads T head_dims
if you observe this is,number of batches -> out of number of batches -> what are the number of heads that performs parallel -> and inside that each heads we perform the matrix multiplication 
and we got the new Q and K (we just make them view in the way we wanted so i was referring this as new,not any other new matrix which comes from air 😂️)

and we can now check the shape of the
Q - > (B,num_heads,T,head_dims)
K - > (B,num_heads,T,head_dims)
so how does this matrix multiplications works ? 
so we need to transpose the dimensions of the K
so we are tranpose those last two dimensions of the K

and then we do the multiplication -> (T,head_dims) @ (head_dim,T) => (T,T)
and this is how we end up with the (T,T) which we have seen earlier 

so we are at a phase of the intermediate
and then once the scores are done we need to mask it right
because in training it can see the next words but in the inference we can't let the model to see netx tokens which are in future so we will make them to the -inf
and then after that we can use the softmax so the e^-inf will goes to the 1/e^inf and then it would be 0,so softmax can make it 0 
then we can calculate the ones with the probabilities and then after this we can strat multiplying wiht the Values
and then we would get the attention out

softmax is the function which works in this way  ==> e^x/sum(e^x) so it was like same probability,like individual values by the sum of the all values but we are using this e^x function
in an exmaple we can take the 
okay as we discussed the tokens for the future ones are gonna become -inf right so this is what we discuss now
so for the row 1 -> imagine for the first token it is the 0.6 and it can't see other right so they comes to the -inf
so after softmax the shape is still (B,H,T,T)
so if we did consider the T=4
and now we can have the softamx values of the e^0.6/e^0.6 0 0 0 and then sum is the e^0.6
and then sum is 1 right
so we need to take the dim as the -1 because,actually the last dim which is the keys dimension ad it is the one which actually calculates the sum
so it is the dim -1 we use for the sum,isnt it the keys sections only rigt we are using for the sum 


in the prev steps we have make the transpose right and we will now revert it and then make this blocks contiguous and then view these as the B T D
because we need to convert them back to the shape we did actually taken them from

and now it is the all the heads are sitting side by side with their own features
like B T D as the total dims are the 512 (for ex...) and then we can take the heads as the 8 right
so [0....63][64.....127][]....[.....511] so these are sitting side by side without any mixing of the info which all have gathered and then
now we use the final projections to make this clear

out -> shape(B,T,D)

proj_out -> shape(B,D,D)

multiplication goes in this way
(T,D) @ (D,D)
now taking a single row
for the first token
0 1 2 3 4 5 6 ............512 this is for the first token and this multiplies with the @ cols of that first proj matrix 
so each of the 512 features are gonna multiplied to the proj_matrix and then comes to the single features by addig what they all gathered 
and then we will get same 512 features again

so this final piece is to integrate the all the pieces which have worked independently
as these are all started random and then ,they all are trained and then we can get the correct matrix weights so they can give the predictions and then find the loss and then adjusts the values
so after many training steps they will become better and then they mill merge better

Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V

Q - > query (what am i looking for ??)
K - > key   (what do i provide..)
V - > value (what information i gave)

Q @ K.T -> similarity these are the scores
softmax -> it convert the scores to the probabilities

/sqrt(d_k) - > to prevent the softmax saturation [d_k is the dimensions of the head]





The argument diagonal controls which diagonal to consider.
If diagonal = 0, all elements on and above the main diagonal are retained. A positive value excludes just as many diagonals above the main diagonal
and similarly a negative value includes just as many diagonals below the main diagonal.
The main diagonal are the set of indices {(i,i)}{(i,i)} for i∈[0,min⁡{d1,d2}−1]i∈[0,min{d1​,d2​}−1] where d1,d2d1​,d2​ are the dimensions of the matrix.




attention backward pass
the loss comes from the next layer and reaches the attention layer
we can consider it as the grad_out
the shape of the grad_out is B,T,D (same shape of the output)

grad = self.proj_out.backward(grad_out)
now the grad came from the backward of that
and then before step of the proj_out we did the transpose and view right
so we need to reshape it:
B,T,D = grad.shape
grad.view(B,T,self.num_heads,self.head_dims)
and transpose it as we did the same in the forward pass right

next attn_scores = attn_scores @ V

attn_scores shape -> B,H,T,T
shape of V? okay we can derive it right now:
shape of w_v -> (D,D) 
x -> shape(B,T,D) (because x is toe+poe)
V -> x @ w_v (B,T,D)
this is before splitting D

after splitting we will be left with (B,T,num_heads,head_dims)
after getting transposed we will get -> (B,num_heads,T,head_dims)
shape of V is derived


grad_attn -> we need to find the gradients for two parts:
one is softmax and the other is V
for softmax we can use:
e^x / sum(e^x) terms

attn_scores = attn_scores @ V
at this step when we need to perform this:
y = x @ w
dL/dx = dL/dy * dy/dx
so grad_attn is what we need now.
we have the grad which is used in the view in the exact way we needed, 
and dy/dx is what we need to find.
at this part for some time V becomes constant and we find the softmax gradient.

out[i, k] = Σⱼ attn_scores[i, j] * V[j, k]

the reason for the sum over j is because j dim is multiplied with j dim and summed across all j's:
attn -> B H T T 
v -> B H T d_k
so T is j here and d_k is k.

for the backward pass we use:

∂L/∂attn_scores[i, j] = Σₖ (∂L/∂out[i, k]) × (∂out[i, k]/∂attn_scores[i, j])

∂L/∂attn_scores[i, j] = grad_out @ ∂out[i, k]/∂attn_scores[i, j]

out[i, k] = Σⱼ attn_scores[i, j] × V[j, k]
differentiating with respect to attn_scores leaves V[j, k]:

∂L/∂attn_scores[i, j] = Σₖ grad_out[i, k] × V[j, k]

so we use V.T


s1 = exp(x1) / (exp(x1) + exp(x2))
s2 = exp(x2) / (exp(x1) + exp(x2))

ds1/dx1:
use quotient rule (v u' - u v') / v^2

u = exp(x1), u' = exp(x1)
v = exp(x1) + exp(x2), v' = exp(x1)

we get s1 * (1 - s1)


Derivative   Result
∂s1/∂x1     s1 * (1 - s1)
∂s1/∂x2    -s1 * s2
∂s2/∂x1    -s2 * s1
∂s2/∂x2     s2 * (1 - s2)


∂s_i / ∂x_j = s_i * (δ_ij - s_j)

where δ_ij = 1 if i == j, else 0

In words:
If i == j: s_i * (1 - s_i) (diagonal)
If i != j: -s_i * s_j (off-diagonal)


now we derive grad_V:
out = attn_scores @ V

dL/dV = dL/dout * dout/dV

s_i = exp(x_i) / Σⱼ exp(x_j)
∂s_i/∂x_j = s_i * (δ_ij - s_j)

grad_V[j, k] = Σᵢ grad_out[i, k] * attn_scores[i, j]

Now look at the shapes:
grad_out[i, k] → index i (row), k (column)
attn_scores[i, j] → index i (row), j (column)

Matrix multiplication rule:
When you do A @ B:
(rows, cols) @ (cols, other)
The index that gets summed over must be the column of A and row of B

We want:
Sum over i (the index that appears in BOTH terms)
attn_scores[i, j] has i as row
grad_out[i, k] has i as row

So we need to transpose one of them to make i the column!
Transpose attn_scores:
attn_scores.T → shape (T, T) with indices [j, i]
Now attn_scores.T[j, i] = attn_scores[i, j]

Now multiply:
grad_V[j, k] = Σᵢ attn_scores.T[j, i] * grad_out[i, k]

This is exactly matrix multiplication:
grad_V = attn_scores.T @ grad_out


so even going backward for the
grad_Q and then grad_k
which gives the scores = Q @ K.transpose()

S = Q @ K.T


S[i, j] = Σₚ Q[i, p] * K[j, p]
Where:

i = query position
j = key position
p = head dimension (the one we sum over)

we need the
∂L/∂Q[i, p] = Σⱼ (∂L/∂S[i, j]) * (∂S[i, j] / ∂Q[i, p])
we now the grad_attn already from the backward
and we need the derivation
∂S[i, j] / ∂Q[i, p] = ?
it is the K[j,p]
sp substitute
grad_Q = grad_scores @ K[j,p]


the main eqn is this 
S[i, j] = Σₚ Q[i, p] * K[j, p]

so we need to find the
∂L/∂K[j, p] = Σⱼ (∂L/∂S[i, j]) * (∂S[i, j] / ∂K[i, p])

grad_K = relation between the grad_scores and ∂S[i, j] / ∂K[i, p]

∂S[i, j] / ∂K[i, p] for this we get the Q[i, p]
as the shape of the grad_scores is rhe (i,j) right
and then,we need to get the

scores(i,j) and Q(i,p) we need to get the (j,p)
so it is the grad_scores.T @ grad_Q

```
x (input) → goes to THREE different paths:
              ↓
         ┌────┼────┐
         ↓    ↓    ↓
        w_q  w_k  w_v
         ↓    ↓    ↓
         Q    K    V


One input x creates THREE different outputs!

grad_Q ← from attention (gradient w.r.t Q)
grad_K ← from attention (gradient w.r.t K)  
grad_V ← from attention (gradient w.r.t V)
         ↓    ↓    ↓
        w_q  w_k  w_v
         ↓    ↓    ↓
      grad_x_from_Q
      grad_x_from_K
      grad_x_from_V
         ↓    ↓    ↓
         └────┼────┘
              ↓
           grad_x (TOTAL)

```
